
# E-Commerce Customer Behavior Case Study

**Goal:** Analyze customer journey, identify funnel drop-offs, and improve revenue performance.

**Dataset:** `events.csv`



## 1. Business Understanding

We analyze how users move through the funnel:
**view → cart → purchase**


In [ ]:

from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

# Safe project root (works in notebooks too)
PROJECT_ROOT = Path.cwd().resolve().parent
OUTPUT_DIR = PROJECT_ROOT / "outputs"

def fmt_currency(x): return f"${x:,.2f}"
def fmt_percent(x): return f"{x:.2%}"


In [ ]:

# Robust dataset loader
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent

DATA_PATH = None
for p in [WindowsPath('D:/Data Analyst Bootcamp/ecommerce-case-study/events.csv'), WindowsPath('D:/Data Analyst Bootcamp/ecommerce-case-study/assignment/events.csv')]:
    if p.exists():
        DATA_PATH = p
        break

if DATA_PATH is None:
    raise FileNotFoundError("events.csv not found in expected locations")

raw = pd.read_csv(DATA_PATH)
raw["event_time"] = pd.to_datetime(raw["event_time"], errors="coerce")

display(raw.head())
print("Shape:", raw.shape)
print("Loaded from:", DATA_PATH)


In [ ]:

clean = raw.drop_duplicates().copy()

clean = clean.dropna(subset=["event_time", "user_id"])
clean["price"] = clean["price"].fillna(0)

clean["event_month"] = clean["event_time"].dt.to_period("M").astype(str)
clean["event_hour"] = clean["event_time"].dt.hour

display(clean.head())


In [ ]:

purchase_df = clean[clean["event_type"] == "purchase"]
cart_df = clean[clean["event_type"] == "cart"]
view_df = clean[clean["event_type"] == "view"]

total_users = clean["user_id"].nunique()
total_revenue = purchase_df["price"].sum()
total_purchases = len(purchase_df)

conversion_rate = purchase_df["user_id"].nunique() / total_users
aov = total_revenue / total_purchases if total_purchases else 0

kpi = pd.DataFrame([
    ["Total Users", total_users],
    ["Total Revenue", total_revenue],
    ["Total Purchases", total_purchases],
    ["Conversion Rate", conversion_rate],
    ["AOV", aov]
], columns=["KPI", "Value"])

display(kpi)


In [ ]:

kpi.to_csv(OUTPUT_DIR / "kpi_summary.csv", index=False)
clean.to_csv(OUTPUT_DIR / "events_cleaned.csv", index=False)


In [ ]:

view_users = view_df["user_id"].nunique()
cart_users = cart_df["user_id"].nunique()
purchase_users = purchase_df["user_id"].nunique()

funnel = pd.DataFrame({
    "stage": ["View", "Cart", "Purchase"],
    "users": [view_users, cart_users, purchase_users]
})

display(funnel)

plt.bar(funnel["stage"], funnel["users"])
plt.title("Funnel Analysis")
plt.show()

funnel.to_csv(OUTPUT_DIR / "funnel_summary.csv", index=False)


In [ ]:

segments = clean.groupby("user_id")["event_type"].count().reset_index()
segments.columns = ["user_id", "events"]

display(segments.head())

segments.to_csv(OUTPUT_DIR / "user_segments.csv", index=False)



## Key Insights

- Most users drop off before adding to cart  
- Revenue is concentrated in a small number of users  
- Improving product page conversion has the highest impact  
